<a href="https://colab.research.google.com/github/zsabro/Batch66/blob/main/HW_task_multiple_agents_and_functions_with_handoffs_v003.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai-agents -q # Install the open-AI-agents SDK package to import liabraries
!pip install nest_asyncio -q # Install nest_asyncio to fix event loop issue


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.9/116.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.7 MB/s eta 0:00:00


In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Allow nested event loops in notebook environments

from agents import Agent, Runner, AsyncOpenAI, set_default_openai_client, set_tracing_disabled, set_default_openai_api
from google.colab import userdata

# Replace with your actual Gemini API key
gemini_api_key = userdata.get('gemini_api_key')
set_tracing_disabled(True)
set_default_openai_api("chat_completions")

# Set up connection to Gemini AI
external_client = AsyncOpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
set_default_openai_client(external_client)

# Create Math Agent for math-related questions
math_agent: Agent = Agent(
    name="Math_Agent",
    instructions="You are a math expert. Answer any math-related questions, such as calculations or explaining math concepts, in a clear and simple way.",
    handoff_description="Handles math-related questions, including calculations, algebra, geometry, and math concepts.",
    model="gemini-2.0-flash"
)

# Create History Agent for history-related questions
history_agent: Agent = Agent(
    name="History_Agent",
    instructions="You are a history expert. Answer any history-related questions, such as events or historical figures, in a clear and simple way.",
    handoff_description = "Handles history-related questions, including historical events, figures, and timelines.",
    model="gemini-2.0-flash"
)

# Create Triage Agent to delegate to other agents
triage_agent: Agent = Agent(
    name="Triage_Agent",
    instructions="You are a triage assistant. Your job is to read the user's prompt and delegate it to the appropriate agent based on their handoff descriptions. If the prompt is about math, hand it off to Math_Agent. If it’s about history, hand it off to History_Agent. If the prompt is irrelevant (not about math or history), respond with: 'Sorry, but I can only answer questions about math or history.'",
    handoffs=[math_agent, history_agent],
    model="gemini-2.0-flash"
)

# Function to process the prompt through Triage Agent
def process_prompt(user_prompt):
    # Run the Triage Agent, which will either hand off or respond
    result = Runner.run_sync(triage_agent, user_prompt)
    return result.final_output

In [ ]:
# Test the system with different prompts
test_prompts = [
    "Calculate 5 + 3",
    "Who was Cleopatra?",
    "What's the weather like?"
]

for prompt in test_prompts:
    print(f"Prompt: {prompt}")
    print(f"Response: {process_prompt(prompt)}\n")

Prompt: Calculate 5 + 3
Response: 5 + 3 = 8


Prompt: Who was Cleopatra?
Response: Cleopatra was the last active ruler of the Ptolemaic Kingdom of Egypt. Think of her as a queen who ruled Egypt a long, long time ago. She's famous for a few key things:

*   **Her intelligence and political skills:** She wasn't just a pretty face. Cleopatra was smart, spoke multiple languages, and was a skilled negotiator. She used these skills to try to maintain Egypt's independence and power in a world dominated by the Roman Republic.

*   **Her relationships with powerful Romans:** She famously had relationships with Julius Caesar and later with Mark Antony. These relationships were both romantic and political, helping her to secure Egypt's position. She had a son with Caesar (Caesarion) and three children with Antony.

*   **Her tragic end:** After being defeated by Octavian (later Emperor Augustus) in a civil war, she and Antony committed suicide. This marked the end of the Ptolemaic Kingdom, and Eg